# 05 - Data Transfer Readiness

Checkpoint 5: inspect saved Data Transfer install, registration, smoke request, delivery, ingestion, validation, and next-action artifacts. This notebook does not submit jobs, authenticate, or read credentials.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "outputs" / "data_transfer"

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None

def show_status(rows):
    return pd.DataFrame(rows)

## Hybrid Architecture

REST path -> bounded/candidate extraction -> known-ID enrichment -> shared internal tables -> diagnostics/dashboard drilldowns

Data Transfer path -> smoke delivery -> local raw files -> ingestion/validation -> full-night request decision -> shared internal tables -> population-level science products

REST remains active; Data Transfer is the planned route for validated complete-night products.

## Fink Client Install Guide

In [ ]:
install_status = load_json(OUT / "fink_client_install_status.json")
if install_status:
    display(show_status([
        {"field": "python_executable", "value": install_status.get("python_executable")},
        {"field": "python_version", "value": install_status.get("python_version")},
        {"field": "fink_client_installed", "value": install_status.get("fink_client_installed")},
        {"field": "fink_client_version", "value": install_status.get("fink_client_version")},
        {"field": "recommended_install_command", "value": install_status.get("recommended_install_command")},
    ]))
else:
    print("Run scripts/install_fink_client_helper.py")

## Setup Status

In [ ]:
setup_status = load_json(OUT / "setup_status.json")
if setup_status:
    display(pd.DataFrame(setup_status["checks"]))
else:
    print("Run scripts/check_fink_data_transfer_setup.py")

## Registration/Auth Checklist

In [ ]:
registration = OUT / "REGISTRATION_AUTH_CHECKLIST.md"
print(registration.read_text() if registration.exists() else "Run scripts/write_registration_checklist.py")

## Full-Night Candidate Request Draft

In [ ]:
request = load_json(OUT / "request_drafts" / "data_transfer_request.json")
if request:
    display(show_status([
        {"field": "profile", "value": request.get("profile")},
        {"field": "target_startdate", "value": request.get("target_startdate")},
        {"field": "target_stopdate", "value": request.get("target_stopdate")},
        {"field": "timezone", "value": request.get("timezone")},
        {"field": "output_format", "value": request.get("output_format")},
        {"field": "all_alerts", "value": request.get("scope", {}).get("all_alerts")},
        {"field": "dry_run", "value": request.get("confirmation", {}).get("dry_run")},
        {"field": "allow_submit", "value": request.get("confirmation", {}).get("allow_submit")},
        {"field": "risk", "value": request.get("risk", {}).get("level")},
    ]))
else:
    print("Run scripts/prepare_data_transfer_request.py")

## Smoke Request Draft

In [ ]:
smoke_request = load_json(OUT / "request_drafts" / "SMOKE_DATA_TRANSFER_REQUEST.json")
if smoke_request:
    display(show_status([
        {"field": "profile", "value": smoke_request.get("profile")},
        {"field": "target_startdate", "value": smoke_request.get("target_startdate")},
        {"field": "target_stopdate", "value": smoke_request.get("target_stopdate")},
        {"field": "output_format", "value": smoke_request.get("output_format")},
        {"field": "all_alerts", "value": smoke_request.get("scope", {}).get("all_alerts")},
        {"field": "completeness_claim_allowed", "value": smoke_request.get("scope", {}).get("completeness_claim_allowed")},
        {"field": "allow_submit", "value": smoke_request.get("confirmation", {}).get("allow_submit")},
        {"field": "risk", "value": smoke_request.get("risk", {}).get("level")},
    ]))
else:
    print("Run scripts/prepare_data_transfer_smoke_request.py")

## Client Probe

In [ ]:
probe = load_json(OUT / "client_probe.json")
if probe:
    display(show_status([
        {"field": "package_installed", "value": probe.get("package_installed")},
        {"field": "cli_available", "value": probe.get("cli_available")},
        {"field": "version_known", "value": probe.get("version_known")},
        {"field": "authentication_known", "value": probe.get("authentication_known")},
        {"field": "job_submission_attempted", "value": probe.get("job_submission_attempted")},
        {"field": "credentials_recorded", "value": probe.get("credentials_recorded")},
    ]))
    display(pd.DataFrame(probe.get("probes", [])))
else:
    print("Run scripts/probe_fink_client_capabilities.py")

## Delivery Inspection

In [ ]:
delivery = load_json(OUT / "delivery_inspection.json")
if delivery:
    display(show_status([
        {"field": "delivery_dir", "value": delivery.get("delivery_dir")},
        {"field": "file_count", "value": delivery.get("file_count")},
        {"field": "empty", "value": delivery.get("empty")},
    ]))
    display(pd.DataFrame(delivery.get("files", [])))
else:
    print("Run scripts/inspect_data_transfer_delivery.py")

## Delivery Drop-Zones

In [ ]:
dropzones = []
for name in ["smoke_delivery", "full_night"]:
    path = ROOT / "data" / "raw" / "data_transfer" / name
    files = [item for item in path.glob("*") if item.is_file() and item.name != ".gitkeep"] if path.exists() else []
    dropzones.append({"zone": name, "path": path, "exists": path.exists(), "file_count": len(files)})
display(pd.DataFrame(dropzones))

## Smoke Ingestion Status

In [ ]:
smoke_runs = sorted((OUT / "smoke_delivery").glob("*/smoke_manifest.json")) if (OUT / "smoke_delivery").exists() else []
if smoke_runs:
    smoke_manifest = load_json(smoke_runs[-1])
    display(pd.DataFrame([smoke_manifest]))
else:
    print("No smoke ingestion run found. Run scripts/run_data_transfer_smoke_ingestion.py after placing smoke delivery files.")

## Validation Status

In [ ]:
validation_files = sorted((OUT).glob("**/*validation_report.json"))
if validation_files:
    validation = load_json(validation_files[-1])
    display(pd.DataFrame(validation["checks"]))
else:
    print("No validation run found. After delivery/ingestion, run scripts/validate_data_transfer_delivery.py")

## Next Manual Action

In [ ]:
next_action = load_json(OUT / "data_transfer_next_action.json")
if next_action:
    display(show_status([
        {"field": "decision", "value": next_action.get("decision")},
        {"field": "user_confirmation_required", "value": next_action.get("user_confirmation_required")},
        {"field": "full_night_request_allowed", "value": next_action.get("full_night_request_allowed")},
        {"field": "next_action", "value": next_action.get("next_action")},
    ]))
else:
    print("Run scripts/report_data_transfer_next_action.py")

## Readiness Checklist

In [ ]:
rows = [
    {"item": "registration needed", "status": "manual"},
    {"item": "fink-client installed", "status": bool(probe and probe.get("package_installed")) if 'probe' in globals() else None},
    {"item": "request prepared", "status": request is not None if 'request' in globals() else False},
    {"item": "smoke request prepared", "status": smoke_request is not None if 'smoke_request' in globals() else False},
    {"item": "delivery present", "status": bool(delivery and not delivery.get("empty")) if 'delivery' in globals() else False},
    {"item": "ingestion ready", "status": True},
    {"item": "validation ready", "status": True},
    {"item": "complete-night claim allowed", "status": False},
]
pd.DataFrame(rows)